In [55]:
import pandas as pd
import xlwings as xw
from xlwings import constants

In [56]:
reviews = pd.read_csv('recipes_model/reviews_sample.csv')
recipes = pd.read_csv('recipes_model/recipes_sample.csv')

recipes = recipes[['id', 'name', 'minutes', 'submitted', 'description', 'n_ingredients']]

print(recipes.head())

      id                                      name  minutes   submitted  \
0  44123     george s at the cove  black bean soup       90  2002-10-25   
1  67664        healthy for them  yogurt popsicles       10  2003-07-26   
2  38798              i can t believe it s spinach       30  2002-08-29   
3  35173                      italian  gut busters       45  2002-07-27   
4  84797  love is in the air  beef fondue   sauces       25  2004-02-23   

                                         description  n_ingredients  
0  an original recipe created by chef scott meska...           18.0  
1  my children and their friends ask for my homem...            NaN  
2            these were so go, it surprised even me.            8.0  
3  my sister-in-law made these for us at a family...            NaN  
4  i think a fondue is a very romantic casual din...            NaN  


In [57]:
print(len(recipes), len(reviews))

recipes_sample = recipes.sample(frac=0.05, random_state=42)
reviews_sample = reviews.sample(frac=0.05, random_state=42)

print(len(recipes_sample), len(reviews_sample))

with pd.ExcelWriter('recipes.xlsx') as writer:
    recipes_sample.to_excel(writer, sheet_name='Рецепты', index=False)
    reviews_sample.to_excel(writer, sheet_name='Отзывы', index=False)

30000 126696
1500 6335


In [58]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

wb = load_workbook('recipes.xlsx')
recipes_sheet = wb['Рецепты']
reviews_sheet = wb['Отзывы']

recipes_sheet['G1'] = 'seconds_assign'
for row in range(2, recipes_sheet.max_row + 1):
    minutes = recipes_sheet[f'C{row}'].value
    recipes_sheet[f'G{row}'] = minutes * 60 if minutes else 0

In [59]:
recipes_sheet['H1'] = 'seconds_formula'
for row in range(2, recipes_sheet.max_row + 1):
    recipes_sheet[f'H{row}'] = f'=C{row}*60'


In [60]:
review_counts = reviews_sample['recipe_id'].value_counts().reset_index()
review_counts.columns = ['id', 'n_reviews']

recipes_merged = recipes_sample.merge(review_counts, on='id', how='left').fillna(0)

recipes_sheet['I1'] = 'n_reviews'
for idx, row in enumerate(dataframe_to_rows(recipes_merged[['n_reviews']], index=False, header=False), 2):
    recipes_sheet[f'I{idx}'] = row[0]

In [61]:
green = PatternFill(start_color='86D080', end_color='86D080', fill_type='solid')
yellow = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
red = PatternFill(start_color='FF0000', end_color='FF0000', fill_type='solid')

for row in recipes_sheet.iter_rows(min_row=2, max_col=3, max_row=recipes_sheet.max_row):
    cell = row[2]
    value = cell.value
    if value < 5:
        cell.fill = green
    elif 5 <= value <= 10:
        cell.fill = yellow
    else:
        cell.fill = red

In [62]:
for col in ['G', 'H', 'I']:
    header = recipes_sheet[f'{col}1']
    header.font = Font(bold=True)
    header.alignment = Alignment(horizontal='center')

In [63]:
def validate():
    valid_ids = set(recipes_sample['id'].astype(str))
    
    for row in reviews_sheet.iter_rows(min_row=2):
        recipe_id = str(row[1].value)
        rating = row[3].value
        
        if not (0 <= rating <= 5) or recipe_id not in valid_ids:
            for cell in row:
                cell.fill = red

validate()

wb.save('recipes.xlsx')